# Frankie Explicit Generator — RealVis XL + IPAdapter FaceID + ADetailer

**T3 / OF-grade** batch: 16 explicit photoreal scenes, face+hand fix pass.

Runtime → **Run all** → ~18 min → 16 explicit Frankie shots.

In [ ]:
!nvidia-smi | head -15

In [ ]:
# ── ComfyUI + custom nodes (IPAdapter + Impact Pack for ADetailer) ──
import os
if not os.path.exists('/content/ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus'):
    !git clone -q https://github.com/cubiq/ComfyUI_IPAdapter_plus /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-Impact-Pack'):
    !git clone -q https://github.com/ltdrdata/ComfyUI-Impact-Pack /content/ComfyUI/custom_nodes/ComfyUI-Impact-Pack
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-Impact-Subpack'):
    !git clone -q https://github.com/ltdrdata/ComfyUI-Impact-Subpack /content/ComfyUI/custom_nodes/ComfyUI-Impact-Subpack
!pip install -q insightface onnxruntime-gpu opencv-python-headless segment-anything ultralytics
print('ComfyUI + Impact Pack ready')

In [ ]:
# ── Models: RealVis XL 5.0 + IPAdapter + CLIP-H + InsightFace + face_yolov8m ──
!mkdir -p /content/ComfyUI/models/checkpoints /content/ComfyUI/models/ipadapter /content/ComfyUI/models/loras /content/ComfyUI/models/clip_vision /content/ComfyUI/models/insightface/models/buffalo_l /content/ComfyUI/models/ultralytics/bbox
CKPT = '/content/ComfyUI/models/checkpoints/realvis_xl_v5.safetensors'
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 1_000_000_000:
    !rm -f {CKPT}
    !wget --show-progress -O {CKPT} https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors
if not os.path.exists('/content/ComfyUI/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin'):
    !wget -q --show-progress -O /content/ComfyUI/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin
    !wget -q --show-progress -O /content/ComfyUI/models/loras/ip-adapter-faceid-plusv2_sdxl_lora.safetensors https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors
if not os.path.exists('/content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors'):
    !wget -q --show-progress -O /content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors
if not os.path.exists('/content/ComfyUI/models/insightface/models/buffalo_l/det_10g.onnx'):
    !wget -q -O /tmp/buffalo_l.zip https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip
    !unzip -q -o /tmp/buffalo_l.zip -d /content/ComfyUI/models/insightface/models/buffalo_l
if not os.path.exists('/content/ComfyUI/models/ultralytics/bbox/face_yolov8m.pt'):
    !wget -q -O /content/ComfyUI/models/ultralytics/bbox/face_yolov8m.pt https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8m.pt
!ls -lh /content/ComfyUI/models/checkpoints

In [ ]:
import requests
FRANKIE_REF_URL = 'https://raw.githubusercontent.com/veyrapay/frankie-assets/main/frankie-ref.png'
os.makedirs('/content/ComfyUI/input', exist_ok=True)
with open('/content/ComfyUI/input/frankie-ref.png', 'wb') as f:
    f.write(requests.get(FRANKIE_REF_URL).content)
print('Frankie ref staged')

In [ ]:
# ── Restart ComfyUI server clean (load Impact Pack nodes too) ──
!pkill -f 'main.py' 2>/dev/null; sleep 3
import subprocess, time, requests as r
subprocess.Popen(['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188', '--disable-auto-launch'], cwd='/content/ComfyUI')
for i in range(90):
    try:
        r.get('http://127.0.0.1:8188/system_stats', timeout=2)
        print(f'ComfyUI up after {i*2}s')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('ComfyUI did not start')
ni = r.get('http://127.0.0.1:8188/object_info').json()
print('IPAdapterFaceID:', 'IPAdapterFaceID' in ni)
print('FaceDetailer:', 'FaceDetailer' in ni)
print('UltralyticsDetectorProvider:', 'UltralyticsDetectorProvider' in ni)

In [ ]:
import json, uuid, time, requests as r
POS_PREFIX = 'RAW photo, photograph, photorealistic, sharp focus, 8k uhd, dslr, natural skin texture, film grain, freckles, dark wavy hair, gold hoop earrings, sleeve tattoos, fit body, '
NEGATIVE = 'cartoon, anime, drawing, painting, illustration, 3d render, cgi, deformed, bad anatomy, extra fingers, mutated hands, watermark, text, signature, blurry, low quality, ugly, plastic skin, oversaturated, airbrushed, child, young'
SCENES = [
    'fully nude lying on white sheets, legs spread, looking at camera, intimate amateur photo, bedroom',
    'fully nude bathroom mirror selfie, full body visible, one hand on hip, sultry expression',
    'fully nude in shower, water on skin, eyes closed, head back, steamy bathroom',
    'fully nude masturbating in bed, hand between legs, eyes half closed, biting lip, intimate POV',
    'fully nude on her knees on the bed, looking up at camera, hands on thighs, sultry',
    'fully nude bent over showing back and ass, looking over shoulder at camera, smirking',
    'fully nude on a beach at sunset, sitting on towel, legs slightly apart, ocean behind',
    'fully nude in a hot tub, water at waist, breasts above water, candid amateur photo',
    'topless wearing only thong panties, full body mirror selfie, bedroom',
    'wearing only an unbuttoned mens dress shirt open showing breasts and panties, morning bed',
    'topless pulling down jeans showing tan lines, hip cocked, bedroom',
    'fully nude doggy style position on bed, looking back at camera, intimate POV',
    'fully nude squatting with legs spread, photoreal amateur photo, soft window light',
    'fully nude touching her own breasts, eyes on camera, biting lip, bedroom',
    'topless on a balcony at night, city lights behind, leaning on railing',
    'fully nude post-shower wrapped only in towel half-fallen, mirror selfie',
]

def build_workflow(prompt, seed):
    """Pipeline: RealVis XL + IPAdapter FaceID → KSampler → ADetailer face restore → save."""
    return {
        '1': {'class_type': 'CheckpointLoaderSimple', 'inputs': {'ckpt_name': 'realvis_xl_v5.safetensors'}},
        '2': {'class_type': 'LoadImage', 'inputs': {'image': 'frankie-ref.png'}},
        '3': {'class_type': 'IPAdapterUnifiedLoaderFaceID', 'inputs': {'model': ['1', 0], 'preset': 'FACEID PLUS V2', 'lora_strength': 0.6, 'provider': 'CUDA'}},
        '4': {'class_type': 'IPAdapterFaceID', 'inputs': {'model': ['3', 0], 'ipadapter': ['3', 1], 'image': ['2', 0], 'weight': 0.85, 'weight_faceidv2': 1.0, 'weight_type': 'linear', 'combine_embeds': 'concat', 'start_at': 0.0, 'end_at': 1.0, 'embeds_scaling': 'V only'}},
        '5': {'class_type': 'CLIPTextEncode', 'inputs': {'text': prompt, 'clip': ['1', 1]}},
        '6': {'class_type': 'CLIPTextEncode', 'inputs': {'text': NEGATIVE, 'clip': ['1', 1]}},
        '7': {'class_type': 'EmptyLatentImage', 'inputs': {'width': 832, 'height': 1216, 'batch_size': 1}},
        '8': {'class_type': 'KSampler', 'inputs': {'model': ['4', 0], 'positive': ['5', 0], 'negative': ['6', 0], 'latent_image': ['7', 0], 'seed': seed, 'steps': 30, 'cfg': 6.0, 'sampler_name': 'dpmpp_2m', 'scheduler': 'karras', 'denoise': 1.0}},
        '9': {'class_type': 'VAEDecode', 'inputs': {'samples': ['8', 0], 'vae': ['1', 2]}},
        '10': {'class_type': 'UltralyticsDetectorProvider', 'inputs': {'model_name': 'bbox/face_yolov8m.pt'}},
        '11': {'class_type': 'FaceDetailer', 'inputs': {'image': ['9', 0], 'model': ['1', 0], 'clip': ['1', 1], 'vae': ['1', 2], 'positive': ['5', 0], 'negative': ['6', 0], 'bbox_detector': ['10', 0], 'guide_size': 384, 'guide_size_for': True, 'max_size': 1024, 'seed': seed + 1, 'steps': 20, 'cfg': 6.0, 'sampler_name': 'dpmpp_2m', 'scheduler': 'karras', 'denoise': 0.4, 'feather': 5, 'noise_mask': True, 'force_inpaint': True, 'bbox_threshold': 0.5, 'bbox_dilation': 10, 'bbox_crop_factor': 3.0, 'sam_detection_hint': 'center-1', 'sam_dilation': 0, 'sam_threshold': 0.93, 'sam_bbox_expansion': 0, 'sam_mask_hint_threshold': 0.7, 'sam_mask_hint_use_negative': 'False', 'drop_size': 10, 'wildcard': '', 'cycle': 1, 'inpaint_model': False, 'noise_mask_feather': 20}},
        '12': {'class_type': 'SaveImage', 'inputs': {'images': ['11', 0], 'filename_prefix': 'frankie_full'}},
    }

client_id = str(uuid.uuid4())
results = []
for i, scene in enumerate(SCENES, 1):
    prompt = POS_PREFIX + scene
    wf = build_workflow(prompt, 42 + i * 13)
    print(f'\n[{i}/{len(SCENES)}] {scene}')
    sub = r.post('http://127.0.0.1:8188/prompt', json={'prompt': wf, 'client_id': client_id}).json()
    if 'error' in sub:
        print('  submit error:', json.dumps(sub.get('error'))[:300])
        print('  node_errors:', json.dumps(sub.get('node_errors', {}))[:600])
        continue
    pid = sub['prompt_id']
    img_path = None
    for poll in range(120):
        time.sleep(3)
        h = r.get(f'http://127.0.0.1:8188/history/{pid}').json()
        if pid in h:
            entry = h[pid]
            status = entry.get('status', {})
            if status.get('status_str') == 'error':
                print('  EXEC ERROR:', json.dumps(status.get('messages', []))[:800])
                break
            outs = entry.get('outputs', {}).get('12', {}).get('images', [])
            if outs:
                f = outs[0]
                img_url = f'http://127.0.0.1:8188/view?filename={f["filename"]}&subfolder={f["subfolder"]}&type={f["type"]}'
                img_path = f'/content/full_{i:02d}.png'
                with open(img_path, 'wb') as fh:
                    fh.write(r.get(img_url).content)
                print(f'  saved {img_path}')
                break
    if img_path:
        results.append((scene, img_path))
print(f'\nDone. {len(results)}/{len(SCENES)} succeeded.')

In [ ]:
from IPython.display import display, Markdown
from PIL import Image
for scene, path in results:
    display(Markdown(f'**{scene}**'))
    display(Image.open(path).resize((512, 768)))

In [ ]:
!cd /content && zip -q frankie_full_batch.zip full_*.png
from google.colab import files
files.download('/content/frankie_full_batch.zip')